# EDA: Реєстр платників ПДВ України

**Опис:** первинний огляд та підготовка даних реєстру платників ПДВ (Open Data).

**Мета:** перевірити структуру, якість даних і підготувати базову візуалізацію для подальшого аналізу.

## Імпорти
Стандартні бібліотеки для аналізу та візуалізації.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## Конфігурація
Налаштування відображення та стилю графіків.

In [ ]:
pd.set_option("display.max_columns", None)
plt.style.use("ggplot")

## Завантаження даних
Читаємо CSV з обробкою кодування та помилкових рядків.

In [ ]:
data_path = Path("..") / "data" / "raw" / "pdv_actual_28-08-2019.csv"

df = pd.read_csv(
    data_path,
    sep=";",
    encoding="utf-8",
    on_bad_lines="skip",
)

df["dat_term"] = df["dat_term"].replace("null", pd.NA)
df["dat_reestr"] = pd.to_datetime(df["dat_reestr"], format="%d.%m.%Y", errors="coerce")

## Базовий огляд
Швидко перевіряємо перші рядки, типи полів та пропуски.

In [ ]:
print("Data head:")
df.head()

In [ ]:
print("Data info:")
df.info()

print("\nData missing values:")
df.isnull().sum()

## Візуалізація
Приклад: кількість реєстрацій за роками.

In [ ]:
registrations_by_year = (
    df.dropna(subset=["dat_reestr"])
    .assign(year=lambda x: x["dat_reestr"].dt.year)
    .groupby("year")
    .size()
    .sort_index()
 )

plt.figure(figsize=(10, 4))
registrations_by_year.plot(kind="bar", color="steelblue")
plt.title("Кiлькiсть реєстрацiй за роками")
plt.xlabel("Рiк")
plt.ylabel("Кiлькiсть")
plt.tight_layout()
plt.show()

## Гіпотеза 1: Сезонність реєстрацій

**Припущення:** Реєстрація нових платників ПДВ має чітко виражену сезонність із піками на початку кожного кварталу (січень, квітень, липень, жовтень).

In [ ]:
df_with_dates = df.dropna(subset=["dat_reestr"]).copy()
df_with_dates["month"] = df_with_dates["dat_reestr"].dt.month
df_with_dates["quarter"] = df_with_dates["dat_reestr"].dt.quarter

registrations_by_month = df_with_dates.groupby("month").size()

print("Реєстрації за місяцями:")
print(registrations_by_month)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(registrations_by_month.index, registrations_by_month.values, color="steelblue")
axes[0].set_xlabel("Місяць")
axes[0].set_ylabel("Кількість реєстрацій")
axes[0].set_title("Розподіл реєстрацій за місяцями")
axes[0].set_xticks(range(1, 13))

registrations_by_quarter = df_with_dates.groupby("quarter").size()
axes[1].bar(registrations_by_quarter.index, registrations_by_quarter.values, color="coral")
axes[1].set_xlabel("Квартал")
axes[1].set_ylabel("Кількість реєстрацій")
axes[1].set_title("Розподіл реєстрацій за кварталами")
axes[1].set_xticks(range(1, 5))

plt.tight_layout()
plt.show()

print("\nАналіз гіпотези:")
print("Очікування: піки на початку кварталів (січень, квітень, липень, жовтень - місяці 1, 4, 7, 10)")
print(f"Реальність: найвищий місяць - {registrations_by_month.idxmax()} (місяцевих реєстрацій: {registrations_by_month.max()})")
print(f"Найнижчий місяць - {registrations_by_month.idxmin()} (місяцевих реєстрацій: {registrations_by_month.min()})")


## Гіпотеза 2: Зміна популярності організаційно-правових форм

**Припущення:** Популярність форми "Приватне підприємство" (ПП) серед нових платників ПДВ катастрофічно знизилася на користь "Товариств з обмеженою відповідальністю" (ТОВ) після 2010 року.

In [ ]:
df_forms = df.dropna(subset=["dat_reestr"]).copy()
df_forms["year"] = df_forms["dat_reestr"].dt.year

def extract_form(name):
    name_upper = str(name).upper()
    if "ТОВ" in name_upper:
        return "ТОВ"
    elif "ПП" in name_upper or "ПРИВАТНЕ ПІДПРИЄМСТВО" in name_upper:
        return "ПП"
    elif '"ЗАТ"' in name_upper or "ЗАТ" in name_upper:
        return "ЗАТ"
    elif "АТ" in name_upper or "АКЦІОНЕРНЕ" in name_upper:
        return "АТ"
    elif "ООО" in name_upper:
        return "ООО"
    else:
        return "Інше"

df_forms["form"] = df_forms["name"].apply(extract_form)

forms_by_year = df_forms.groupby(["year", "form"]).size().unstack(fill_value=0)

print("Розподіл за роками та формами:")
print(forms_by_year)

before_2010 = df_forms[df_forms["year"] < 2010].groupby("form").size()
after_2010 = df_forms[df_forms["year"] >= 2010].groupby("form").size()

print("\nДо 2010 року:")
print(before_2010)
print(f"Всього: {before_2010.sum()}")

print("\nПісля 2010 року:")
print(after_2010)
print(f"Всього: {after_2010.sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

forms_by_year.plot(ax=axes[0], marker="o")
axes[0].set_title("Динаміка популярності форм за роками")
axes[0].set_xlabel("Рік")
axes[0].set_ylabel("Кількість реєстрацій")
axes[0].legend(title="Форма", loc="best")
axes[0].grid(True, alpha=0.3)

periods = ["До 2010", "Після 2010"]
pp_values = [before_2010.get("ПП", 0), after_2010.get("ПП", 0)]
tov_values = [before_2010.get("ТОВ", 0), after_2010.get("ТОВ", 0)]

x = range(len(periods))
width = 0.35

axes[1].bar([i - width/2 for i in x], pp_values, width, label="ПП", color="steelblue")
axes[1].bar([i + width/2 for i in x], tov_values, width, label="ТОВ", color="coral")
axes[1].set_title("Популярність ПП vs ТОВ до/після 2010")
axes[1].set_ylabel("Кількість реєстрацій")
axes[1].set_xticks(x)
axes[1].set_xticklabels(periods)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nАналіз гіпотези")
if after_2010.get("ПП", 0) < before_2010.get("ПП", 0):
    print(f"ПП дійсно знизилась: {before_2010.get('ПП', 0)} → {after_2010.get('ПП', 0)}")
else:
    print(f"ПП не знизилась або зросла: {before_2010.get('ПП', 0)} → {after_2010.get('ПП', 0)}")

if after_2010.get("ТОВ", 0) > before_2010.get("ТОВ", 0):
    print(f"ТОВ дійсно зросла: {before_2010.get('ТОВ', 0)} → {after_2010.get('ТОВ', 0)}")
else:
    print(f"ТОВ не зросла: {before_2010.get('ТОВ', 0)} → {after_2010.get('ТОВ', 0)}")
